In [1]:
RERUN = False          # DNA_RERUN=1 in the environment turns the GPU rerun cells on
from _common import *
from _dna_helpers import *
import matplotlib.pyplot as plt

RERUN = rerun_flag(RERUN)
P = paper()

# 03 · Cyclizability-guided DNA design (Table 1, Fig 3, Tables 4–5)

Paper §3.1 and App. B.1. Table 1 runs (`scripts/sample_yeast_c0_guidance_hist.py`; ported as
`dmfm.experiments.sample_c0_guidance`): parent-disjoint L = 50 base flow (`yeast_parent_a_L50_dfm_dit_h192_b4/best.pt`),
value-gradient guidance whose posterior samples come from **GLASS** (`nfe_value = 4` steps of the base flow's GLASS
velocity, mc = 8; `args.json` has only the base-flow `ckpt`, no dMFM checkpoint, and `docs/rebuttal_cyc.md` of the DNA
repo calls it "GLASS posterior guidance (MC=8)"), guidance_frac = 8, reward σ = 0.15, reward_scale = 0.5, guidance
window t ∈ [0.5, 0.95], guide regressor trained on `a_train`, oracle trained on `b_train`. **Discrepancy (for the
authors):** §3.1 presents this as dMFM guidance. Everything below is rebuilt on CPU from `results/dna/`
(see `results/dna/PROVENANCE.md`).

| Paper item | Status |
|---|---|
| Table 1 | **switched to the n = 1,000 run** (authors' decision 2026-09-29); every cell recomputed here and written to `results/dna/c0_guidance/table1_n1000_summary.json`. The submitted PDF printed the n = 32 pilot; those numbers are reproduced exactly and shown alongside. |
| §3.1 text | "1,000 pairs" now matches the table; MAE sentence becomes 1.17 → 0.15 (+1) and 0.83 → 0.12 (−1) |
| Fig 3 | schematic; only the caption numbers are data (Table 11, recomputed here) — unchanged |
| Figs 7–8 | **moved** to `notebooks/09_dna_regenerated.ipynb` (dna-regen agent; regenerated) |
| Tables 4–5 | reproduced exactly from the June `split65k`-model records recovered from a Codex transcript (the run dirs were purged); App. B.1 attributes them to the parent-disjoint model, the records say `split65k`. The parent-disjoint n = 1,000 recomputation is shown next to them and differs |


## Table 1 — oracle evaluation, n = 1,000 pairs per target (revised table)

Runs `workdir/yeast_parent_glass_n1000_target{1p0,m1p0}_g8.0_seed0/` (Slurm jobs 48651189 / 48651190, 2026-09-25):
same flags as the submitted n = 32 pilot except `--n_samples 1000`, generated with the reconstructed
`sample_yeast_c0_guidance_hist.py` and oracle-scored in the same job. Per pair *i* the unguided and guided sequences
share the initial noise. Oracle scores (reverse-complement-averaged `b_train` oracle) are in
`oracle/oracle_per_sample.csv`. MAE = mean |oracle − y⋆|; paired improvement = mean(|u − y⋆| − |g − y⋆|).

**CI method (same as the submitted Table 1):** percentile bootstrap of the mean over pairs, 2,000 resamples,
`numpy.random.default_rng(0).choice(values, (2000, n))` re-seeded for every statistic, 2.5 / 97.5 % quantiles,
float32 scores — exactly `scripts/score_yeast_c0_oracle.py::bootstrap_mean` with `--bootstrap 2000 --seed 0`
(`_dna_helpers.table1_cells`). `build_table1_summary()` below (re)writes
`results/dna/c0_guidance/table1_n1000_summary.json`, the file the revised tex takes its numbers from.

`compare()` rows: *revised* = the values the revised manuscript prints (`paper_values.json → table1_revised_n1000`).

In [2]:
T1 = build_table1_summary(write=True)          # -> results/dna/c0_guidance/table1_n1000_summary.json
t1_n1000, t1_pilot = T1["n1000"], T1["submitted_n32"]["recomputed"]
print("max |recomputed - oracle_summary.csv written by the original scoring job|:",
      max(v["_max_abs_diff_vs_stored_oracle_summary"] for v in t1_n1000.values()))

CELLS = ["oracle_mean_unguided", "oracle_mean_guided", "unguided_mae", "guided_mae", "paired_improvement"]


def table1_rows(ref: dict, ours: dict, note: str = "") -> list[dict]:
    rows = []
    for k in ["+1", "-1"]:
        p, o = ref[f"target_{k}"], ours[k]
        for c in CELLS:
            rows.append(dict(item=f"y*={k} {c}", paper=p[c], ours=o[c], dec=3, note=note))
            if c + "_ci" in p:
                for j, side in enumerate(["low", "high"]):
                    rows.append(dict(item=f"y*={k} {c} CI {side}", paper=p[c + "_ci"][j], ours=o[c + "_ci"][j], dec=3, note=note))
    return rows


show(cmp(table1_rows(P["table1_revised_n1000"], t1_n1000, "revised Table 1 (n=1000)"))).rename(columns={"paper": "revised"})

max |recomputed - oracle_summary.csv written by the original scoring job|: 2.220446049250313e-16
status counts: {'match': 22}


,item,revised,ours,status,tol,tol_src,note
0,y*=+1 oracle_mean_unguided,-0.171,-0.171460,match,0.0005,atol,revised Table 1 (n=1000)
1,y*=+1 oracle_mean_guided,0.917,0.917096,match,0.0005,atol,revised Table 1 (n=1000)
2,y*=+1 unguided_mae,1.174,1.173698,match,0.0005,atol,revised Table 1 (n=1000)
3,y*=+1 unguided_mae CI low,1.149,1.148523,match,0.0005,atol,revised Table 1 (n=1000)
4,y*=+1 unguided_mae CI high,1.200,1.200498,match,0.0005,atol,revised Table 1 (n=1000)
5,y*=+1 guided_mae,0.148,0.148292,match,0.0005,atol,revised Table 1 (n=1000)
6,y*=+1 guided_mae CI low,0.141,0.141194,match,0.0005,atol,revised Table 1 (n=1000)
7,y*=+1 guided_mae CI high,0.156,0.155643,match,0.0005,atol,revised Table 1 (n=1000)
8,y*=+1 paired_improvement,1.025,1.025406,match,0.0005,atol,revised Table 1 (n=1000)
9,y*=+1 paired_improvement CI low,0.999,0.999189,match,0.0005,atol,revised Table 1 (n=1000)


In [3]:
pd.DataFrame(T1["table1_n1000_formatted"]).T.assign(
    **{"frac pairs improved": [f"{t1_n1000[k]['frac_pairs_improved']:.3f}" for k in ["+1", "-1"]]})

,target,mean_oracle_score_unguided,mean_oracle_score_guided,unguided_mae,guided_mae,paired_improvement,frac pairs improved
+1,+1,-0.171,0.917,"1.174 [1.149, 1.200]","0.148 [0.141, 0.156]","1.025 [0.999, 1.052]",0.983
-1,-1,-0.171,-0.949,"0.833 [0.806, 0.857]","0.119 [0.114, 0.125]","0.713 [0.687, 0.740]",0.971


### Revised (n = 1,000) next to the submitted (n = 32) table
The submitted PDF printed the n = 32 pilot (`workdir/yeast_parent_glass_pilot_target{1p0,m1p0}_g8.0_seed0/`, jobs
35242279 / 35242278). Its first 32 unguided sequences are identical to the n = 1,000 run's; guided samples are not
bit-reproducible (nondeterministic CUDA backward). At n = 1,000 the point estimates move by ≤ 0.035 and the CIs
shrink ≈ 5×. Every n = 1,000 point estimate lies inside the corresponding pilot CI, and every n = 1,000 CI is nested in
the pilot's except the guided MAE at y⋆ = −1 (n = 1,000: 0.119 [0.114, 0.125]; pilot: 0.153 [0.118, 0.190]). The
check is computed in the cell after the table.

In [4]:
side = []
for k in ["+1", "-1"]:
    for name, src in [("revised: n=1000", t1_n1000[k]), ("submitted PDF", P["table1_c0_guidance"][f"target_{k}"]),
                      ("recomputed n=32 pilot", t1_pilot[k])]:
        side.append({"y*": k, "source": name,
                     "oracle mean unguided": f"{src['oracle_mean_unguided']:.3f}", "oracle mean guided": f"{src['oracle_mean_guided']:.3f}",
                     "unguided MAE": fmt_ci(src["unguided_mae"], src["unguided_mae_ci"]),
                     "guided MAE": fmt_ci(src["guided_mae"], src["guided_mae_ci"]),
                     "paired improvement": fmt_ci(src["paired_improvement"], src["paired_improvement_ci"]),
                     "n pairs": src.get("n_pairs", 32)})

nest = []
for k in ["+1", "-1"]:
    for c in ["unguided_mae", "guided_mae", "paired_improvement"]:
        a, b = t1_n1000[k], t1_pilot[k]
        nest.append({"y*": k, "cell": c, "|n1000 - n32|": abs(a[c] - b[c]),
                     "n1000 point in pilot CI": b[c + "_ci"][0] <= a[c] <= b[c + "_ci"][1],
                     "n1000 CI nested in pilot CI": b[c + "_ci"][0] <= a[c + "_ci"][0] and a[c + "_ci"][1] <= b[c + "_ci"][1],
                     "CI width ratio pilot/n1000": (b[c + "_ci"][1] - b[c + "_ci"][0]) / (a[c + "_ci"][1] - a[c + "_ci"][0])})
display(pd.DataFrame(side))
pd.DataFrame(nest).round(3)

,y*,source,oracle mean unguided,oracle mean guided,unguided MAE,guided MAE,paired improvement,n pairs
0,+1,revised: n=1000,-0.171,0.917,"1.174 [1.149, 1.200]","0.148 [0.141, 0.156]","1.025 [0.999, 1.052]",1000
1,+1,submitted PDF,-0.169,0.912,"1.170 [1.038, 1.291]","0.157 [0.119, 0.196]","1.013 [0.881, 1.142]",32
2,+1,recomputed n=32 pilot,-0.169,0.912,"1.170 [1.038, 1.291]","0.157 [0.119, 0.196]","1.013 [0.881, 1.142]",32
3,-1,revised: n=1000,-0.171,-0.949,"0.833 [0.806, 0.857]","0.119 [0.114, 0.125]","0.713 [0.687, 0.740]",1000
4,-1,submitted PDF,-0.169,-0.916,"0.831 [0.709, 0.964]","0.153 [0.118, 0.190]","0.678 [0.554, 0.816]",32
5,-1,recomputed n=32 pilot,-0.169,-0.916,"0.831 [0.709, 0.964]","0.153 [0.118, 0.190]","0.678 [0.554, 0.816]",32


,y*,cell,|n1000 - n32|,n1000 point in pilot CI,n1000 CI nested in pilot CI,CI width ratio pilot/n1000
0,+1,unguided_mae,0.004,True,True,4.874
1,+1,guided_mae,0.009,True,True,5.316
2,+1,paired_improvement,0.013,True,True,4.912
3,-1,unguided_mae,0.001,True,True,4.969
4,-1,guided_mae,0.033,True,False,6.185
5,-1,paired_improvement,0.035,True,True,4.950


The submitted table is reproduced exactly from the n = 32 pilot (every cell, including the bootstrap CIs):

In [5]:
show(cmp(table1_rows(P["table1_c0_guidance"], t1_pilot, "submitted = n=32 pilot")))

status counts: {'match': 22}


,item,paper,ours,status,tol,tol_src,note
0,y*=+1 oracle_mean_unguided,-0.169,-0.168864,match,0.0005,atol,submitted = n=32 pilot
1,y*=+1 oracle_mean_guided,0.912,0.911989,match,0.0005,atol,submitted = n=32 pilot
2,y*=+1 unguided_mae,1.170,1.169813,match,0.0005,atol,submitted = n=32 pilot
3,y*=+1 unguided_mae CI low,1.038,1.038073,match,0.0005,atol,submitted = n=32 pilot
4,y*=+1 unguided_mae CI high,1.291,1.291420,match,0.0005,atol,submitted = n=32 pilot
5,y*=+1 guided_mae,0.157,0.157120,match,0.0005,atol,submitted = n=32 pilot
6,y*=+1 guided_mae CI low,0.119,0.119309,match,0.0005,atol,submitted = n=32 pilot
7,y*=+1 guided_mae CI high,0.196,0.196123,match,0.0005,atol,submitted = n=32 pilot
8,y*=+1 paired_improvement,1.013,1.012692,match,0.0005,atol,submitted = n=32 pilot
9,y*=+1 paired_improvement CI low,0.881,0.881302,match,0.0005,atol,submitted = n=32 pilot


And the submitted numbers against the n = 1,000 run (MISMATCH expected: this is the change the authors made):

In [6]:
show(cmp(table1_rows(P["table1_c0_guidance"], t1_n1000, "CHANGED: Table 1 switched to n=1000")))

status counts: {'MISMATCH': 22}


,item,paper,ours,status,tol,tol_src,note
0,y*=+1 oracle_mean_unguided,-0.169,-0.171460,MISMATCH,0.0005,atol,CHANGED: Table 1 switched to n=1000
1,y*=+1 oracle_mean_guided,0.912,0.917096,MISMATCH,0.0005,atol,CHANGED: Table 1 switched to n=1000
2,y*=+1 unguided_mae,1.170,1.173698,MISMATCH,0.0005,atol,CHANGED: Table 1 switched to n=1000
3,y*=+1 unguided_mae CI low,1.038,1.148523,MISMATCH,0.0005,atol,CHANGED: Table 1 switched to n=1000
4,y*=+1 unguided_mae CI high,1.291,1.200498,MISMATCH,0.0005,atol,CHANGED: Table 1 switched to n=1000
5,y*=+1 guided_mae,0.157,0.148292,MISMATCH,0.0005,atol,CHANGED: Table 1 switched to n=1000
6,y*=+1 guided_mae CI low,0.119,0.141194,MISMATCH,0.0005,atol,CHANGED: Table 1 switched to n=1000
7,y*=+1 guided_mae CI high,0.196,0.155643,MISMATCH,0.0005,atol,CHANGED: Table 1 switched to n=1000
8,y*=+1 paired_improvement,1.013,1.025406,MISMATCH,0.0005,atol,CHANGED: Table 1 switched to n=1000
9,y*=+1 paired_improvement CI low,0.881,0.999189,MISMATCH,0.0005,atol,CHANGED: Table 1 switched to n=1000


### §3.1 text
Submitted: "For each target, we generate 1,000 guided–unguided pairs" (the printed table was n = 32); "Guidance reduces
oracle MAE from 1.17 to 0.16 for y⋆ = +1 and from 0.83 to 0.15 for y⋆ = −1"; "The guided errors of 0.16 and 0.15 are
comparable to the oracle's own MAE of 0.16"; "not that it attains the target to within 0.15".

Revised (n = 1,000): 1,000 pairs (now true); **1.17 → 0.15** (+1) and **0.83 → 0.12** (−1); "guided errors of **0.15 and
0.12** are comparable to the oracle's own MAE of 0.16"; the "to within 0.15" clause should become "to within 0.12" (or
be rephrased), since both guided errors are now below the oracle's own error. The suggested wording is stored in
`table1_n1000_summary.json → in_text_sec3_1.suggested_sentences`.

In [7]:
rv, sb = P["table1_revised_n1000"]["text_mae_rounded"], P["table1_c0_guidance"]["text_mae_rounded"]
o_mae = T1["fig3_caption_and_table11"]["oracle"]["mae"]
rows = [dict(item="pairs per target (revised)", paper=P["table1_revised_n1000"]["n_pairs_per_target"], ours=t1_n1000["+1"]["n_pairs"], atol=0),
        dict(item="pairs per target (submitted text) vs submitted table", paper=P["table1_c0_guidance"]["text_n_pairs_per_target"],
             ours=t1_pilot["+1"]["n_pairs"], atol=0, note="KNOWN (resolved by the switch): submitted table was the n=32 pilot")]
for k in ["+1", "-1"]:
    for which in ["unguided", "guided"]:
        rows.append(dict(item=f"revised text y*={k} {which} MAE", paper=rv[f"{k}_{which}"], ours=t1_n1000[k][f"{which}_mae"], dec=2))
        rows.append(dict(item=f"submitted text y*={k} {which} MAE (vs n=32)", paper=sb[f"{k}_{which}"], ours=t1_pilot[k][f"{which}_mae"], dec=2))
rows.append(dict(item="text: oracle's own held-out MAE", paper=rv["oracle_own_mae"], ours=o_mae, dec=2, note="Table 11 oracle MAE 0.1620"))
show(cmp(rows))

status counts: {'match': 10, 'MISMATCH': 1}


,item,paper,ours,status,tol,tol_src,note
0,pairs per target (revised),1000.00,1000.000000,match,0.000,atol,
1,pairs per target (submitted text) vs submitted table,1000.00,32.000000,MISMATCH,0.000,atol,KNOWN (resolved by the switch): submitted table was the n=32 pilot
2,revised text y*=+1 unguided MAE,1.17,1.173698,match,0.005,atol,
3,submitted text y*=+1 unguided MAE (vs n=32),1.17,1.169813,match,0.005,atol,
4,revised text y*=+1 guided MAE,0.15,0.148292,match,0.005,atol,
5,submitted text y*=+1 guided MAE (vs n=32),0.16,0.157120,match,0.005,atol,
6,revised text y*=-1 unguided MAE,0.83,0.832561,match,0.005,atol,
7,submitted text y*=-1 unguided MAE (vs n=32),0.83,0.831136,match,0.005,atol,
8,revised text y*=-1 guided MAE,0.12,0.119226,match,0.005,atol,
9,submitted text y*=-1 guided MAE (vs n=32),0.15,0.152637,match,0.005,atol,


## Table 1 with the dMFM as the posterior sampler (authors' decision 2026-09-29)

The submitted Table 1 and its n = 1,000 rerun above both take their posterior samples from **GLASS on
the base DFM** (`dmfm.experiments.sample_c0_guidance`; §3.1 presents it as dMFM guidance, which is the
discrepancy noted at the top). The decision is to make Table 1 the **dMFM** run: the identical Table 1
protocol with the 4 GLASS Euler steps replaced by 4 composed dMFM flow-map steps
(`dmfm.experiments.sample_c0_guidance_dmfm --paired --posterior dmfm`, wrapper
`scripts/dna/table1_c0_guidance_dmfm.sbatch`, n = 1,000; everything else — reward, MC = 8 frozen
per-sample pool, guidance_frac 8, t ∈ [0.5, 0.95], 64 trajectory steps, noise pairing — unchanged).

The full-size run is dna-core's job array **49277977**, which publishes
`results/dna/c0_guidance/table1_dmfm_n1000_summary.json`. The cell below prints that table next to the
GLASS n = 1,000 one as soon as the file exists, and says PENDING until then. `--paired --posterior
glass` reproduces the printed-PDF code path exactly (`tests/dna/test_api.py::test_paired_glass_equals_table1_sampler`),
so the two rows are comparable cell by cell.

In [8]:
T1_DMFM = RESULTS / "dna" / "c0_guidance" / "table1_dmfm_n1000_summary.json"
if T1_DMFM.exists():
    T1D = json.loads(T1_DMFM.read_text())
    cd = T1D.get("n1000") or T1D.get("dmfm") or T1D
    ok = all(isinstance(cd.get(k), dict) and "guided_mae" in cd.get(k, {}) for k in ("+1", "-1"))
    if not ok:
        print(f"{T1_DMFM.name} exists but not in the {{'+1': {{...}}, '-1': {{...}}}} layout this cell expects.")
        print("top-level keys:", list(T1D)[:20])
    else:
        side = []
        for k in ["+1", "-1"]:
            for c in CELLS:
                side.append({"y*": k, "cell": c,
                             "dMFM posterior (n=1000)": round(cd[k][c], 4),
                             "dMFM 95% CI": fmt_ci(cd[k][c], cd[k][c + "_ci"]) if c + "_ci" in cd[k] else "",
                             "GLASS posterior (n=1000)": round(t1_n1000[k][c], 4),
                             "GLASS 95% CI": fmt_ci(t1_n1000[k][c], t1_n1000[k][c + "_ci"]) if c + "_ci" in t1_n1000[k] else ""})
        display(pd.DataFrame(side))
        # same protocol, only the posterior sampler differs: differences are the finding, not an error
        show(cmp([dict(item=f"y*={k} {c}", paper=t1_n1000[k][c], ours=cd[k][c], dec=3,
                       note="GLASS vs dMFM posterior, n=1000, identical protocol")
                  for k in ["+1", "-1"] for c in CELLS])).rename(
            columns={"paper": "GLASS posterior", "ours": "dMFM posterior"})
else:
    print("Table 1 with the dMFM posterior: PENDING (not yet published).")
    print("  full size   : sbatch scripts/dna/table1_c0_guidance_dmfm.sbatch   (dna-core job array 49277977)")
    print("  expected    : results/dna/c0_guidance/table1_dmfm_n1000_summary.json")
    print("  code path   : python -m dmfm.experiments.sample_c0_guidance_dmfm --paired --posterior dmfm ...")
    print("  Until then the table above (GLASS posterior, n=1000) is what this notebook reports, and the")
    print("  RERUN cells at the end rerun BOTH posteriors at n=8 so the two can already be compared.")

Table 1 with the dMFM posterior: PENDING (not yet published).
  full size   : sbatch scripts/dna/table1_c0_guidance_dmfm.sbatch   (dna-core job array 49277977)
  expected    : results/dna/c0_guidance/table1_dmfm_n1000_summary.json
  code path   : python -m dmfm.experiments.sample_c0_guidance_dmfm --paired --posterior dmfm ...
  Until then the table above (GLASS posterior, n=1000) is what this notebook reports, and the
  RERUN cells at the end rerun BOTH posteriors at n=8 so the two can already be compared.


## Fig 3 caption numbers (guide / oracle accuracy on the 58 held-out parents)

Fig 3 is a schematic. Its caption quotes Pearson r and MAE of the two regressors on the 8,294 held-out test
windows; recomputed from the stored test predictions (full table in notebook 02, Table 11).

In [9]:
rows = []
for role in ["guide", "oracle"]:
    d = load_csv(f"dna/c0_regressors/{role}/test_predictions.csv")
    mae = float(np.abs(d.prediction - d.label).mean())
    r = float(np.corrcoef(d.prediction, d.label)[0, 1])
    rows += [dict(item=f"{role} Pearson r", paper=P["fig3_caption"][f"{role}_pearson"], ours=r, dec=2),
             dict(item=f"{role} MAE", paper=P["fig3_caption"][f"{role}_mae"], ours=mae, dec=3),
             dict(item=f"{role} n test windows", paper=P["fig3_caption"]["n_test_windows"], ours=len(d), atol=0)]
rows.append(dict(item="held-out test parents", paper=P["fig3_caption"]["n_test_parents"],
                 ours=len(load_json("dna/training/parent_split_seed0.json")["parent_split"]["test"]), atol=0))
show(cmp(rows))

status counts: {'match': 7}


,item,paper,ours,status,tol,tol_src,note
0,guide Pearson r,0.880,0.877412,match,0.0050,atol,
1,guide MAE,0.159,0.158772,match,0.0005,atol,
2,guide n test windows,8294.000,8294.000000,match,0.0000,atol,
3,oracle Pearson r,0.870,0.872417,match,0.0050,atol,
4,oracle MAE,0.162,0.161985,match,0.0005,atol,
5,oracle n test windows,8294.000,8294.000000,match,0.0000,atol,
6,held-out test parents,58.000,58.000000,match,0.0000,atol,


## Figs 7–8 — moved to `notebooks/09_dna_regenerated.ipynb`

The data behind the submitted Fig 7 (tilt of the C0 distribution at target 0.3 for three guidance strengths) and
Fig 8 (per-step gradient norm / value / predicted endpoint score of one guided sample) were lost; both figures are
regenerated by the dna-regen agent in `notebooks/09_dna_regenerated.ipynb` (results in `results/dna/regen/`). The
analog histogram the first round drew here from the n = 1,000 runs is kept as a file only
(`notebooks/figures/dna_fig7_analog_n1000.png`, not a paper figure).

## Tables 4–5 — diversity of C0-steered samples

**Where the printed numbers come from.** Tables 4–5 (and Table 6, notebook 09) come from the June 2026 analysis
`workdir/yeast_split_dmfm_c0_diversity_2026-06-09/diversity_summary.csv` (tracked copy
`docs/results/yeast_split_dmfm_c0_diversity_summary_2026-06-09.csv`), made by `scripts/analyze_yeast_c0_diversity.py`
and `scripts/write_yeast_split_diversity_markdown.py`. All of these were purged. Their content survives in a Codex
transcript (2026-06-09/10 and 2026-07-25) and is stored in `results/dna/june_split65k_recovered/`:
- `…_2026-06-09.head5.csv`: `head -5` of the tracked CSV = header + the four generated rows at full precision;
- `…_2026-06-09.selected_columns.txt`: a pandas print of 8 columns for all 8 rows (the 4 real-data reference rows
  are only available here, at 5–6 decimals);
- `clean_split_sequence_validation.md`: the report written from that CSV (its markdown tables are the paper's
  Tables 4–6 cell for cell; the only record of the `pairwise median` column for the split reference rows);
- `yeast_split_dmfm_c0_diversity_2026-06-02.diversity_summary.csv`: the complete earlier (2026-06-02) summary of the
  same sample sets, before the split-reference rows were added.

**Discrepancy (for the authors).** App. B.1 says these samples come from "the parent-disjoint L = 50 dMFM trained on
the 230 a-parents" and that novelty is measured against "the 58 held-out test parents". The records show the June
**`split65k`** models (random 65,000 / 8,000 / 9,368 window split of `data/yeast_mid50.pt`; teacher
`yeast_mid50_dfm_dit_split65k_…/last.pt`, student `yeast_mid50_dmfm_split65k_esd_resume_…/last.ckpt`), guidance by
`dinko/C0free_torch.pt` with mc = 16, and split-train / split-test = that random split (see
`samples_target_{p1,m1}_2026-06-02.args.json`). A parent-disjoint recomputation is shown further below.

In [10]:
J = "dna/june_split65k_recovered"
jgen = load_csv(f"{J}/yeast_split_dmfm_c0_diversity_summary_2026-06-09.head5.csv")
sel_lines = (RESULTS / J / "yeast_split_dmfm_c0_diversity_summary_2026-06-09.selected_columns.txt").read_text().splitlines()
jsel = pd.DataFrame([l.split() for l in sel_lines[1:]], columns=sel_lines[0].split())
for c in jsel.columns[1:]:
    jsel[c] = jsel[c].astype(float)
jsel = jsel.set_index("set")


def md_table(path: Path, header_start: str) -> pd.DataFrame:
    """Parse the first markdown table whose header row starts with `header_start`."""
    lines = path.read_text().splitlines()
    i = next(k for k, l in enumerate(lines) if l.startswith(header_start))
    rows = []
    for l in lines[i:]:
        if not l.startswith("|"):
            break
        rows.append([c.strip() for c in l.strip().strip("|").split("|")])
    return pd.DataFrame(rows[2:], columns=rows[0])


report = RESULTS / J / "clean_split_sequence_validation.md"
jrep4 = md_table(report, "| set | n | exact unique").set_index("set")
jsel

,target_c0,nearest_gen_frac_mean,pairwise_gen_frac_mean,nearest_split_test_frac_mean,exact_unique_frac,novel_exact_vs_split_train_frac,novel_exact_vs_split_test_frac
set,,,,,,,
guided,-1.0,0.41748,0.701028,0.43608,1.0,1.0,1.0
unguided,-1.0,0.52100,0.736424,0.47788,1.0,1.0,1.0
guided,1.0,0.49132,0.737193,0.47690,1.0,1.0,1.0
unguided,1.0,0.52152,0.737013,0.47870,1.0,1.0,1.0
split_test_sample,NaN,0.52010,0.737401,NaN,1.0,NaN,NaN
split_train_sample,NaN,0.52192,0.737335,NaN,1.0,NaN,NaN
split_val_sample,NaN,0.51840,0.737177,NaN,1.0,NaN,NaN
train_sample,NaN,0.52188,0.737699,NaN,1.0,NaN,NaN


### Table 4 — real-data baselines (1,000 windows each, analyzer seed 0), paper vs recovered June records
`exact unique`, `nearest generated` and `pairwise generated` are compared with the recovered pandas print (5–6
decimals); `n` and `pairwise median` exist only in the recovered report (4 decimals).

In [11]:
rows = []
for prow, vals in P["table4_real_baselines"].items():
    if prow.startswith("_"):
        continue
    pv = dict(zip(P["table4_real_baselines"]["_columns"], vals))
    r = jsel.loc[prow]
    rows += [dict(item=f"{prow} n", paper=pv["n"], ours=float(jrep4.loc[prow, "n"]), atol=0, note="recovered report"),
             dict(item=f"{prow} exact unique", paper=pv["exact_unique"], ours=r.exact_unique_frac, dec=4),
             dict(item=f"{prow} nearest generated", paper=pv["nearest_generated"], ours=r.nearest_gen_frac_mean, dec=4),
             dict(item=f"{prow} pairwise generated", paper=pv["pairwise_generated"], ours=r.pairwise_gen_frac_mean, dec=4),
             dict(item=f"{prow} pairwise median", paper=pv["pairwise_median"], ours=float(jrep4.loc[prow, "pairwise median"]), dec=4,
                  note="recovered report")]
t4 = show(cmp(rows))
t4

status counts: {'match': 20}


,item,paper,ours,status,tol,tol_src,note
0,train_sample n,1000.0000,1000.000000,match,0.00000,atol,recovered report
1,train_sample exact unique,1.0000,1.000000,match,0.00005,atol,
2,train_sample nearest generated,0.5219,0.521880,match,0.00005,atol,
3,train_sample pairwise generated,0.7377,0.737699,match,0.00005,atol,
4,train_sample pairwise median,0.7400,0.740000,match,0.00005,atol,recovered report
5,split_train_sample n,1000.0000,1000.000000,match,0.00000,atol,recovered report
6,split_train_sample exact unique,1.0000,1.000000,match,0.00005,atol,
7,split_train_sample nearest generated,0.5219,0.521920,match,0.00005,atol,
8,split_train_sample pairwise generated,0.7373,0.737335,match,0.00005,atol,
9,split_train_sample pairwise median,0.7400,0.740000,match,0.00005,atol,recovered report


### Table 5 — generated sets, paper vs recovered June records (full precision)
Column mapping (analyzer names): guide score mean = `score_mean`; novel train / test = `novel_exact_vs_split_{train,test}_frac`;
nearest train / test = `nearest_split_{train,test}_frac_mean`; nearest gen. = `nearest_gen_frac_mean`; pairwise gen. =
`pairwise_gen_frac_mean`; the two ratios divide by the `split_test_sample` row (pairwise 0.737401, nearest 0.52010).

In [12]:
held = jsel.loc["split_test_sample"]
col5 = {"n": "n_samples", "guide_score_mean": "score_mean", "exact_unique": "exact_unique_frac",
        "novel_train": "novel_exact_vs_split_train_frac", "novel_test": "novel_exact_vs_split_test_frac",
        "nearest_train": "nearest_split_train_frac_mean", "nearest_test": "nearest_split_test_frac_mean",
        "nearest_gen": "nearest_gen_frac_mean", "pairwise_gen": "pairwise_gen_frac_mean"}
june5 = {}
for _, r in jgen.iterrows():
    key = f"{'+1' if r.target_c0 > 0 else '-1'}_{r.set}"
    o = {k: float(r[c]) for k, c in col5.items()}
    o["pairwise_over_heldout"] = o["pairwise_gen"] / held.pairwise_gen_frac_mean
    o["nearest_gen_over_heldout"] = o["nearest_gen"] / held.nearest_gen_frac_mean
    june5[key] = o
rows = []
for key, vals in P["table5_generated_diversity"].items():
    if key.startswith("_"):
        continue
    for c, pv in zip(P["table5_generated_diversity"]["_columns"], vals):
        rows.append(dict(item=f"{key} {c}", paper=pv, ours=june5[key][c], atol=0 if c == "n" else None, dec=None if c == "n" else 4))
t5 = show(cmp(rows))
t5

status counts: {'match': 44}


,item,paper,ours,status,tol,tol_src,note
0,-1_guided n,1000.0000,1000.000000,match,0.00000,atol,
1,-1_guided guide_score_mean,-0.9893,-0.989289,match,0.00005,atol,
2,-1_guided exact_unique,1.0000,1.000000,match,0.00005,atol,
3,-1_guided novel_train,1.0000,1.000000,match,0.00005,atol,
4,-1_guided novel_test,1.0000,1.000000,match,0.00005,atol,
5,-1_guided nearest_train,0.3966,0.396640,match,0.00005,atol,
6,-1_guided nearest_test,0.4361,0.436080,match,0.00005,atol,
7,-1_guided nearest_gen,0.4175,0.417480,match,0.00005,atol,
8,-1_guided pairwise_gen,0.7010,0.701028,match,0.00005,atol,
9,-1_guided pairwise_over_heldout,0.9507,0.950674,match,0.00005,atol,


### App. B.1 prose numbers (repeat Table 4–5 cells)

In [13]:
tx = P["appB1_text"]
rows = [dict(item="+1 guided pairwise", paper=tx["plus1_guided_pairwise"], ours=june5["+1_guided"]["pairwise_gen"], dec=4),
        dict(item="held-out pairwise", paper=tx["heldout_pairwise"], ours=held.pairwise_gen_frac_mean, dec=4),
        dict(item="+1 guided pairwise / held-out", paper=tx["plus1_pairwise_ratio"], ours=june5["+1_guided"]["pairwise_over_heldout"], dec=4),
        dict(item="+1 guided nearest gen", paper=tx["plus1_nearest_gen"], ours=june5["+1_guided"]["nearest_gen"], dec=4),
        dict(item="held-out nearest gen", paper=tx["heldout_nearest_gen"], ours=held.nearest_gen_frac_mean, dec=4),
        dict(item="-1 guided pairwise", paper=tx["minus1_pairwise"], ours=june5["-1_guided"]["pairwise_gen"], dec=4),
        dict(item="-1 guided pairwise / held-out", paper=tx["minus1_pairwise_ratio"], ours=june5["-1_guided"]["pairwise_over_heldout"], dec=4),
        dict(item="-1 guided nearest gen", paper=tx["minus1_nearest_gen"], ours=june5["-1_guided"]["nearest_gen"], dec=4),
        dict(item="-1 guided nearest gen / held-out", paper=tx["minus1_nearest_ratio"], ours=june5["-1_guided"]["nearest_gen_over_heldout"], dec=4),
        dict(item="n per set", paper=tx["n_per_set"], ours=float(jgen.n_samples.min()), atol=0)]
show(cmp(rows))

status counts: {'match': 10}


,item,paper,ours,status,tol,tol_src,note
0,+1 guided pairwise,0.7372,0.737193,match,0.00005,atol,
1,held-out pairwise,0.7374,0.737401,match,0.00005,atol,
2,+1 guided pairwise / held-out,0.9997,0.999718,match,0.00005,atol,
3,+1 guided nearest gen,0.4913,0.491320,match,0.00005,atol,
4,held-out nearest gen,0.5201,0.520100,match,0.00005,atol,
5,-1 guided pairwise,0.7010,0.701028,match,0.00005,atol,
6,-1 guided pairwise / held-out,0.9507,0.950674,match,0.00005,atol,
7,-1 guided nearest gen,0.4175,0.417480,match,0.00005,atol,
8,-1 guided nearest gen / held-out,0.8027,0.802692,match,0.00005,atol,
9,n per set,1000.0000,1000.000000,match,0.00000,atol,


### Consistency of the recovered pieces
The four generated rows agree between the full-precision `head -5` output, the separate pandas print, and the complete
2026-06-02 summary of the same sample sets; the `train_sample` row of 2026-06-02 agrees with the 2026-06-09 print.

In [14]:
j02 = load_csv(f"{J}/yeast_split_dmfm_c0_diversity_2026-06-02.diversity_summary.csv")
chk = []
for _, r in jgen.iterrows():
    key = f"{'+1' if r.target_c0 > 0 else '-1'}_{r.set}"
    s = jsel[(jsel.index == r.set) & (jsel.target_c0 == r.target_c0)].iloc[0]
    r02 = j02[(j02.set == r.set) & (j02.target_c0 == r.target_c0)].iloc[0]
    chk += [dict(item=f"{key} nearest_gen (print, 5 dp)", paper=s.nearest_gen_frac_mean, ours=r.nearest_gen_frac_mean, atol=5e-6),
            dict(item=f"{key} pairwise_gen (print, 6 dp)", paper=s.pairwise_gen_frac_mean, ours=r.pairwise_gen_frac_mean, atol=5e-7),
            dict(item=f"{key} nearest_split_test (print, 5 dp)", paper=s.nearest_split_test_frac_mean, ours=r.nearest_split_test_frac_mean, atol=5e-6)]
    chk += [dict(item=f"{key} {c} (2026-06-02 file)", paper=float(r02[c]), ours=float(r[c]), atol=0)
            for c in ["score_mean", "nearest_train_frac_mean", "nearest_gen_frac_mean", "pairwise_gen_frac_mean", "exact_unique_frac"]]
t02 = j02[j02.set == "train_sample"].iloc[0]
chk += [dict(item="train_sample nearest_gen (2026-06-02 file vs 06-09 print)", paper=jsel.loc["train_sample", "nearest_gen_frac_mean"], ours=float(t02.nearest_gen_frac_mean), atol=5e-6),
        dict(item="train_sample pairwise_gen (2026-06-02 file vs 06-09 print)", paper=jsel.loc["train_sample", "pairwise_gen_frac_mean"], ours=float(t02.pairwise_gen_frac_mean), atol=5e-7)]
show(cmp(chk)).rename(columns={"paper": "other record"})

status counts: {'match': 34}


,item,other record,ours,status,tol,tol_src,note
0,"-1_guided nearest_gen (print, 5 dp)",0.417480,0.417480,match,5.000000e-06,atol,
1,"-1_guided pairwise_gen (print, 6 dp)",0.701028,0.701028,match,5.000000e-07,atol,
2,"-1_guided nearest_split_test (print, 5 dp)",0.436080,0.436080,match,5.000000e-06,atol,
3,-1_guided score_mean (2026-06-02 file),-0.989289,-0.989289,match,0.000000e+00,atol,
4,-1_guided nearest_train_frac_mean (2026-06-02 file),0.392800,0.392800,match,0.000000e+00,atol,
5,-1_guided nearest_gen_frac_mean (2026-06-02 file),0.417480,0.417480,match,0.000000e+00,atol,
6,-1_guided pairwise_gen_frac_mean (2026-06-02 file),0.701028,0.701028,match,0.000000e+00,atol,
7,-1_guided exact_unique_frac (2026-06-02 file),1.000000,1.000000,match,0.000000e+00,atol,
8,"-1_unguided nearest_gen (print, 5 dp)",0.521000,0.521000,match,5.000000e-06,atol,
9,"-1_unguided pairwise_gen (print, 6 dp)",0.736424,0.736424,match,5.000000e-07,atol,


### Parent-disjoint recomputation (the models the rest of the DNA section uses)

The 2026-09-25 analysis of the parent-disjoint n = 1,000 C0 runs (the Table 1 rerun above) with the original
DNA-MFM@187fe7b metric functions (`workdir/rebuttal_parent_c0_diversity_n1000_20260925/`), train = `a_train`,
val = `a_val`, test = `test`. Generated-set statistics (exact uniqueness, nearest-generated and pairwise Hamming,
guide-score mean) are recomputed here from the sequences in `sample_scores.csv`; nearest-train/test distances need
the 82k-window reference data, so their per-sample values are read from `diversity_per_sample.csv` and averaged
here. Real-data baselines are taken from the stored `table6_baselines.csv` (row mapping: `train_sample` ↔
`full_sample`, `split_train_sample` ↔ `a_train_sample`, `split_val_sample` ↔ `a_val_sample`,
`split_test_sample` ↔ `test_sample`). These are **not** the paper's numbers; they are what Tables 4–5 would show
if they were switched to the parent-disjoint models (decision for the authors), so they are shown side by side,
not scored against the paper. One exact identity holds: `train_sample` is a seed-0 draw of 1,000 windows from all
82,368 windows, and the parent-disjoint data file holds the same windows in the same order as the old
`yeast_mid50.pt`, so that row is identical.

In [15]:
base = load_csv("dna/diversity/n1000/table6_baselines.csv").set_index("set")
mapping = {"train_sample": "full_sample", "split_train_sample": "a_train_sample", "split_val_sample": "a_val_sample", "split_test_sample": "test_sample"}
show(cmp([dict(item=f"train_sample {c}", paper=P["table4_real_baselines"]["train_sample"][i], ours=float(base.loc["full_sample", oc]), dec=4,
               note="parent-disjoint full_sample = June train_sample")
          for i, c, oc in [(1, "exact_unique", "exact_unique_frac"), (2, "nearest_generated", "nearest_gen_frac_mean"),
                           (3, "pairwise_generated", "pairwise_gen_frac_mean"), (4, "pairwise_median", "pairwise_gen_frac_median")]]))

status counts: {'match': 4}


,item,paper,ours,status,tol,tol_src,note
0,train_sample exact_unique,1.0000,1.000000,match,0.00005,atol,parent-disjoint full_sample = June train_sample
1,train_sample nearest_generated,0.5219,0.521880,match,0.00005,atol,parent-disjoint full_sample = June train_sample
2,train_sample pairwise_generated,0.7377,0.737699,match,0.00005,atol,parent-disjoint full_sample = June train_sample
3,train_sample pairwise_median,0.7400,0.740000,match,0.00005,atol,parent-disjoint full_sample = June train_sample


In [16]:
side4 = []
for prow, orow in mapping.items():
    pv = dict(zip(P["table4_real_baselines"]["_columns"], P["table4_real_baselines"][prow]))
    ov = base.loc[orow]
    side4.append({"set (paper)": prow, "set (parent-disjoint)": orow,
                  "nearest gen: paper": pv["nearest_generated"], "nearest gen: parent-disjoint": round(float(ov.nearest_gen_frac_mean), 4),
                  "pairwise: paper": pv["pairwise_generated"], "pairwise: parent-disjoint": round(float(ov.pairwise_gen_frac_mean), 4),
                  "median: paper": pv["pairwise_median"], "median: parent-disjoint": round(float(ov.pairwise_gen_frac_median), 4)})
pd.DataFrame(side4)

,set (paper),set (parent-disjoint),nearest gen: paper,nearest gen: parent-disjoint,pairwise: paper,pairwise: parent-disjoint,median: paper,median: parent-disjoint
0,train_sample,full_sample,0.5219,0.5219,0.7377,0.7377,0.74,0.74
1,split_train_sample,a_train_sample,0.5219,0.5200,0.7373,0.7361,0.74,0.74
2,split_val_sample,a_val_sample,0.5184,0.5194,0.7372,0.7385,0.74,0.74
3,split_test_sample,test_sample,0.5201,0.5219,0.7374,0.7381,0.74,0.74


In [17]:
per = load_csv("dna/diversity/n1000/diversity_per_sample.csv")
held_pair = float(base.loc["test_sample", "pairwise_gen_frac_mean"])
held_near = float(base.loc["test_sample", "nearest_gen_frac_mean"])
t5_cols = P["table5_generated_diversity"]["_columns"]
ours5, side5 = {}, []
for t, tv in [("m1p0", -1.0), ("1p0", 1.0)]:
    s = load_csv(f"dna/c0_guidance/n1000/target_{t}/sample_scores.csv")
    for setname in ["guided", "unguided"]:
        seqs = s[f"seq_{setname}"].tolist()
        tok = tokens(seqs)
        pw = float(pairwise_hamming_frac(tok).mean())
        ng = float(nearest_generated_frac(tok).mean())
        pp = per[(per.target_c0 == tv) & (per.set == setname)]
        assert pp.seq.tolist() == seqs
        o = {"n": len(seqs), "guide_score_mean": float(s[setname].mean()), "exact_unique": len(set(seqs)) / len(seqs),
             "novel_train": float((pp.nearest_a_train_hamming > 0).mean()), "novel_test": float((pp.nearest_test_hamming > 0).mean()),
             "nearest_train": float(pp.nearest_a_train_frac.mean()), "nearest_test": float(pp.nearest_test_frac.mean()),
             "nearest_gen": ng, "pairwise_gen": pw, "pairwise_over_heldout": pw / held_pair, "nearest_gen_over_heldout": ng / held_near}
        key = f"{'+1' if tv > 0 else '-1'}_{setname}"
        ours5[key] = o
        for c, pv in zip(t5_cols, P["table5_generated_diversity"][key]):
            side5.append({"row": key, "column": c, "paper (= June split65k)": pv, "parent-disjoint n=1000": round(o[c], 4)})
pd.DataFrame(side5).pivot(index="column", columns="row").reindex(t5_cols)

paper (= June split65k)                                    parent-disjoint n=1000                                   
row                                    +1_guided +1_unguided  -1_guided -1_unguided              +1_guided +1_unguided  -1_guided -1_unguided
column                                                                                                                                       
n                                      1000.0000   1000.0000  1000.0000   1000.0000              1000.0000   1000.0000  1000.0000   1000.0000
guide_score_mean                          1.0010     -0.1766    -0.9893     -0.1692                 0.9986     -0.1767    -0.9932     -0.1767
exact_unique                              1.0000      1.0000     1.0000      1.0000                 1.0000      1.0000     1.0000      1.0000
novel_train                               1.0000      1.0000     1.0000      1.0000                 1.0000      1.0000     1.0000      1.0000
novel_test                                1.0000      1.0000     1.0000      1.0000                 1.0000      1.0000     1.0000      1.0000
nearest_train                             0.4446      0.4466     0.3966      0.4449                 0.4559      0.4583     0.4458      0.4583
nearest_test                              0.4769      0.4787     0.4361      0.4779                 0.4804      0.4817     0.4737      0.4817
nearest_gen                               0.4913      0.5215     0.4175      0.5210                 0.5091      0.5221     0.4960      0.5221
pairwise_gen                              0.7372      0.7370     0.7010      0.7364                 0.7348      0.7374     0.7287      0.7374
pairwise_over_heldout                     0.9997      0.9995     0.9507      0.9987                 0.9956      0.9991     0.9872      0.9991
nearest_gen_over_heldout                  0.9447      1.0027     0.8027      1.0017                 0.9755      1.0003     0.9504      1.0003

Cross-check against the analysis outputs written on 2026-09-25 (`table7_generated.csv`): the generated-set statistics
recomputed above agree with them. (The two unguided parent-disjoint sets are identical — both runs use seed 0 — whereas
the June runs used seeds 0 and 1, hence two different unguided rows in the paper.)

In [18]:
g7 = load_csv("dna/diversity/n1000/table7_generated.csv")
chk = []
for _, r in g7.iterrows():
    key = f"{'+1' if r.target_c0 > 0 else '-1'}_{r.set}"
    for c, oc in [("pairwise_gen", "pairwise_gen_frac_mean"), ("nearest_gen", "nearest_gen_frac_mean"), ("nearest_train", "nearest_a_train_frac_mean"),
                  ("nearest_test", "nearest_test_frac_mean"), ("guide_score_mean", "guide_score_mean"), ("pairwise_over_heldout", "pairwise_vs_test_sample")]:
        chk.append(dict(item=f"{key} {c}", paper=float(r[oc]), ours=ours5[key][c], atol=1e-6))
show(cmp(chk)).rename(columns={"paper": "stored_2026-09-25"})

status counts: {'match': 24}


,item,stored_2026-09-25,ours,status,tol,tol_src,note
0,-1_guided pairwise_gen,0.728695,0.728695,match,0.000001,atol,
1,-1_guided nearest_gen,0.496040,0.496040,match,0.000001,atol,
2,-1_guided nearest_train,0.445820,0.445820,match,0.000001,atol,
3,-1_guided nearest_test,0.473680,0.473680,match,0.000001,atol,
4,-1_guided guide_score_mean,-0.993239,-0.993239,match,0.000001,atol,
5,-1_guided pairwise_over_heldout,0.987237,0.987237,match,0.000001,atol,
6,-1_unguided pairwise_gen,0.737431,0.737431,match,0.000001,atol,
7,-1_unguided nearest_gen,0.522060,0.522060,match,0.000001,atol,
8,-1_unguided nearest_train,0.458260,0.458260,match,0.000001,atol,
9,-1_unguided nearest_test,0.481700,0.481700,match,0.000001,atol,


**Summary for Tables 4–5.** Every printed cell is reproduced from the recovered June `split65k` records. The
parent-disjoint rerun agrees with the qualitative narrative (all samples exact-unique and exact-novel; the −1 guided
set is the more clustered one), but its numbers differ: pairwise / held-out is 0.987 (−1 guided) and 0.996 (+1
guided) vs 0.9507 and 0.9997 in the paper, i.e. the −1 clustering is much weaker for the parent-disjoint model.

## RERUN = True — rerun on one GPU at small n

Every cell below reruns a real experiment of this section through the ported package `src/dmfm`
(the same `dmfm.experiments.*` entry points that `scripts/dna/*.sbatch` call for the full-size
runs), at a small number of samples/states, and prints the small-n result next to the stored
full-size number. They need a CUDA GPU and `source scripts/env.sh`; outputs go to
`outputs/dna/notebook_rerun/` (gitignored). Turn them on without editing the notebook with

```bash
DNA_RERUN=1 jupyter nbconvert --to notebook --execute --ExecutePreprocessor.kernel_name=multimfm \
    --output-dir <somewhere outside the repo> <this notebook>
```

(see `scripts` in the DNA provenance notes, or `python notebooks/_dna_helpers.py rerun <nb>`).
With `RERUN = False` each cell prints one line saying what it would rerun.

**Guided sampling is not bit-reproducible.** The value gradient is accumulated through a
nondeterministic CUDA backward and cuBLAS/cuDNN kernels depend on batch shape and GPU model, so
guided sequences and their scores differ between runs and from the stored runs; they agree
statistically. The *unguided* branch takes no gradient and is deterministic — it must match the
stored sequences exactly, and the cells below check that.

In [19]:
# Table 1 RERUN: paired unguided/guided C0 sampling at L=50 with GLASS value gradients
# (dmfm.experiments.sample_c0_guidance, the flags of scripts/dna/table1_c0_guidance.sbatch) and
# independent-oracle scoring (dmfm.experiments.score_c0_oracle): 8 pairs per target, not 1,000.
if RERUN:
    from dmfm import paths as dpaths

    dev = require_gpu()
    N = 8          # batch_size 8 is part of the RNG stream, so ids 0..7 are the stored run's first batch
    rerun_banner("Table 1 - C0-guided sampling at L=50 + independent-oracle scoring",
                 small=f"{N} pairs per target (one batch, seed 0), guide = a_train, oracle = b_train",
                 stored="1,000 pairs per target (Slurm jobs 48651189 / 48651190)")
    small = {}
    for tag, target in [("1p0", 1.0), ("m1p0", -1.0)]:
        d = rerun_dir("c0_guidance", f"target_{tag}")
        run_experiment("sample_c0_guidance", [
            "--ckpt", dpaths.base_ckpt(50), "--c0_ckpt", dpaths.c0_guide(), "--output_dir", d,
            "--seed", 0, "--n_samples", N, "--batch_size", 8, "--target_c0", target,
            "--reward_sigma", 0.15, "--reward_scale", 0.5, "--mc", 8, "--mc_chunk", 8,
            "--nfe_traj", 64, "--nfe_value", 4, "--t_max", 0.95,
            "--guide_t_start", 0.50, "--guide_t_end", 0.95, "--guidance_frac", 8.0,
            "--coeff_cap", 10.0, "--grad_clip", 10.0])
        run_experiment("score_c0_oracle", [
            "--oracle_ckpt", dpaths.c0_oracle(), "--oracle_model_type", "park_cnn",
            "--run_glob", d / "sample_scores.csv", "--out_dir", d / "oracle",
            "--batch_size", 512, "--bootstrap", 2000, "--seed", 0,
            "--reverse_complement_average", "--device", dev])
        per = pd.read_csv(d / "oracle" / "oracle_per_sample.csv")
        small[tag] = dict(dir=d, target=target, samples=pd.read_csv(d / "sample_scores.csv"),
                          oracle=per, cells=table1_cells(per, target))

    # (1) the deterministic half: unguided sequences and scores must equal the stored n = 1000 run
    det = []
    for tag, key in [("1p0", "+1"), ("m1p0", "-1")]:
        s = small[tag]["samples"].set_index("sample_idx")
        o = small[tag]["oracle"].set_index("sample_idx")
        rs = load_csv(f"dna/c0_guidance/n1000/target_{tag}/sample_scores.csv").set_index("sample_idx").loc[s.index]
        ro = load_csv(f"dna/c0_guidance/n1000/target_{tag}/oracle/oracle_per_sample.csv").set_index("sample_idx").loc[s.index]
        det += [dict(item=f"y*={key} unguided sequences identical (of {len(s)})", paper=len(s), atol=0,
                     ours=int((s.seq_unguided.to_numpy() == rs.seq_unguided.to_numpy()).sum())),
                dict(item=f"y*={key} max |unguided guide score - stored|", paper=0.0, atol=1e-3,
                     ours=float(np.abs(s.unguided.to_numpy() - rs.unguided.to_numpy()).max())),
                dict(item=f"y*={key} max |unguided oracle score - stored|", paper=0.0, atol=1e-3,
                     ours=float(np.abs(o.oracle_unguided.to_numpy() - ro.oracle_unguided.to_numpy()).max()))]
        n_same_guided = int((s.seq_guided.to_numpy() == rs.seq_guided.to_numpy()).sum())
        print(f"y*={key}: guided sequences identical to the stored run: {n_same_guided}/{len(s)} "
              f"(not expected to be 8 - nondeterministic CUDA backward); guided oracle score "
              f"mean {o.oracle_guided.mean():+.4f} here vs {ro.oracle_guided.mean():+.4f} stored on the same ids")
    display(stored_vs_small(det))

    # (2) the Table 1 cells at n = 8 against the n = 1000 run and its bootstrap CI
    rows, side = [], []
    for tag, key in [("1p0", "+1"), ("m1p0", "-1")]:
        c, st = small[tag]["cells"], t1_n1000[key]
        for name in ["oracle_mean_unguided", "oracle_mean_guided", "unguided_mae", "guided_mae", "paired_improvement"]:
            side.append({"y*": key, "cell": name, f"n={N} rerun": round(c[name], 4),
                         "n=1000 stored": round(st[name], 4),
                         "n=1000 95% CI": (fmt_ci(st[name], st[name + "_ci"]) if name + "_ci" in st else "")})
            if name + "_ci" in st:
                rows.append(dict(item=f"y*={key} {name}", stored=round(st[name], 4),
                                 ours=round(c[name], 4), ci_ours=c[name + "_ci"],
                                 note=f"n={c['n_pairs']} vs n={st['n_pairs']}; CI = this run's own bootstrap"))
    display(pd.DataFrame(side))
    display(agree_ci(rows))

    # (3) The sharpest reference available: the stored n=1000 run restricted to THESE 8 ids. Those
    # pairs have the same initial noise and the same frozen MC pool as the rerun, so the unguided
    # cells must agree to float precision and the guided ones differ only through the
    # nondeterministic backward -- unlike the 1,000-pair mean, which differs by subsample noise too.
    same = []
    for tag, key in [("1p0", "+1"), ("m1p0", "-1")]:
        ids = small[tag]["samples"].sample_idx.tolist()
        ref_per = (load_csv(f"dna/c0_guidance/n1000/target_{tag}/oracle/oracle_per_sample.csv")
                   .set_index("sample_idx").loc[ids].reset_index())
        cref, c8 = table1_cells(ref_per, small[tag]["target"]), small[tag]["cells"]
        for name in ["unguided_mae", "guided_mae", "paired_improvement"]:
            same.append(dict(item=f"y*={key} {name}, same 8 ids of the stored run", stored=round(cref[name], 4),
                             ours=round(c8[name], 4), ci_ours=c8[name + "_ci"],
                             note="deterministic" if name == "unguided_mae" else "guided branch: nondeterministic backward"))
    display(agree_ci(same))
    print("\nThe CI above is this 8-pair run's own 2,000-resample bootstrap (the same estimator as Table 1),\n"
          "and the check is whether the stored n=1,000 value falls inside it -- the right direction when the\n"
          "rerun has fewer pairs; the n=1,000 CI itself is ~11x narrower and is shown in the table above.\n"
          "Full size:\n"
          "  TARGET_C0=1.0 sbatch scripts/dna/table1_c0_guidance.sbatch   (and TARGET_C0=-1.0)")
else:
    print("RERUN = False: Table 1 is rebuilt above from results/dna/c0_guidance/n1000/.")
    print("RERUN = True regenerates 8 pairs per target with dmfm.experiments.sample_c0_guidance +")
    print("score_c0_oracle and checks them against the stored n=1000 run (unguided must match exactly).")

RERUN = False: Table 1 is rebuilt above from results/dna/c0_guidance/n1000/.
RERUN = True regenerates 8 pairs per target with dmfm.experiments.sample_c0_guidance +
score_c0_oracle and checks them against the stored n=1000 run (unguided must match exactly).


In [20]:
# Table 1 RERUN, part 2: the same 8 pairs per target with the dMFM as the posterior sampler
# (dmfm.experiments.sample_c0_guidance_dmfm --paired --posterior dmfm = the flags of
# scripts/dna/table1_c0_guidance_dmfm.sbatch), i.e. the run the authors' decision makes Table 1.
if RERUN:
    rerun_banner("Table 1 with the dMFM posterior - 4 composed flow-map steps instead of 4 GLASS Euler steps",
                 small=f"{N} pairs per target, identical protocol to the cell above",
                 stored="n=1000, dna-core job array 49277977 (results/dna/c0_guidance/table1_dmfm_n1000_summary.json)")
    small_d = {}
    for tag, target in [("1p0", 1.0), ("m1p0", -1.0)]:
        d = rerun_dir("c0_guidance_dmfm", f"target_{tag}")
        run_experiment("sample_c0_guidance_dmfm", [
            "--paired", "--posterior", "dmfm",
            "--teacher_ckpt", dpaths.base_ckpt(50), "--student_ckpt", dpaths.dmfm_ckpt(50),
            "--c0_ckpt", dpaths.c0_guide(), "--out_dir", d,
            "--seed", 0, "--n_samples", N, "--batch_size", 8, "--target_c0", target,
            "--reward_sigma", 0.15, "--reward_scale", 0.5, "--mc", 8, "--mc_chunk", 8,
            "--nfe_sample", 64, "--nfe_value", 4, "--t_max", 0.95,
            "--guide_t_start", 0.50, "--guide_t_end", 0.95, "--guidance_frac", 8.0,
            "--coeff_cap", 10.0, "--grad_clip", 10.0])
        run_experiment("score_c0_oracle", [
            "--oracle_ckpt", dpaths.c0_oracle(), "--oracle_model_type", "park_cnn",
            "--run_glob", d / "sample_scores.csv", "--out_dir", d / "oracle",
            "--batch_size", 512, "--bootstrap", 2000, "--seed", 0,
            "--reverse_complement_average", "--device", dev])
        per = pd.read_csv(d / "oracle" / "oracle_per_sample.csv")
        small_d[tag] = dict(dir=d, target=target, samples=pd.read_csv(d / "sample_scores.csv"),
                            oracle=per, cells=table1_cells(per, target))

    # The unguided branch never touches the posterior sampler, so it must equal both the GLASS rerun
    # above and the stored run, sequence for sequence.
    det = []
    for tag, key in [("1p0", "+1"), ("m1p0", "-1")]:
        s = small_d[tag]["samples"].set_index("sample_idx")
        g = small[tag]["samples"].set_index("sample_idx").loc[s.index]
        rs = load_csv(f"dna/c0_guidance/n1000/target_{tag}/sample_scores.csv").set_index("sample_idx").loc[s.index]
        det += [dict(item=f"y*={key} unguided sequences identical to the stored n=1000 run (of {len(s)})",
                     paper=len(s), atol=0, ours=int((s.seq_unguided.to_numpy() == rs.seq_unguided.to_numpy()).sum())),
                dict(item=f"y*={key} unguided sequences identical to the GLASS rerun above (of {len(s)})",
                     paper=len(s), atol=0, ours=int((s.seq_unguided.to_numpy() == g.seq_unguided.to_numpy()).sum()),
                     note="the posterior sampler only enters the guided branch")]
    display(stored_vs_small(det))

    # the two posteriors side by side at n = 8, and against the full-size dMFM run if it is published
    side, rows = [], []
    for tag, key in [("1p0", "+1"), ("m1p0", "-1")]:
        cg, cdm = small[tag]["cells"], small_d[tag]["cells"]
        for c in ["oracle_mean_unguided", "oracle_mean_guided", "unguided_mae", "guided_mae", "paired_improvement"]:
            side.append({"y*": key, "cell": c, f"GLASS posterior (n={N})": round(cg[c], 4),
                         f"dMFM posterior (n={N})": round(cdm[c], 4),
                         f"dMFM 95% CI (n={N})": fmt_ci(cdm[c], cdm[c + "_ci"]) if c + "_ci" in cdm else "",
                         "GLASS n=1000 stored": round(t1_n1000[key][c], 4)})
            if T1_DMFM.exists() and c + "_ci" in cdm:
                _cd = (json.loads(T1_DMFM.read_text()).get("n1000") or {})
                if isinstance(_cd.get(key), dict) and c in _cd[key]:
                    rows.append(dict(item=f"y*={key} {c} (dMFM)", stored=round(_cd[key][c], 4),
                                     ours=round(cdm[c], 4), ci_ours=cdm[c + "_ci"],
                                     note=f"n={N} vs the published dMFM n=1000"))
    display(pd.DataFrame(side))
    if rows:
        display(agree_ci(rows))
    else:
        print("No published dMFM n=1000 summary yet (job array 49277977), so only the two posteriors at "
              f"n={N} are compared above; the GLASS column is the same protocol and the same noise.")
    print("\nFull size: sbatch scripts/dna/table1_c0_guidance_dmfm.sbatch  (POSTERIOR=dmfm, n=1000, sharded)")
else:
    print("RERUN = False: the dMFM-posterior Table 1 is read from results/dna/c0_guidance/"
          "table1_dmfm_n1000_summary.json when it exists (see the Table 1 section above).")
    print("RERUN = True reruns 8 pairs per target with dmfm.experiments.sample_c0_guidance_dmfm")
    print("--paired --posterior dmfm and puts them next to the GLASS-posterior rerun.")

RERUN = False: the dMFM-posterior Table 1 is read from results/dna/c0_guidance/table1_dmfm_n1000_summary.json when it exists (see the Table 1 section above).
RERUN = True reruns 8 pairs per target with dmfm.experiments.sample_c0_guidance_dmfm
--paired --posterior dmfm and puts them next to the GLASS-posterior rerun.


In [21]:
# Tables 4-5 RERUN: the parent-disjoint diversity analysis (dmfm.experiments.analyze_c0_diversity,
# = scripts/dna/tables4_5_c0_diversity.sh) on the two small sample sets generated in the cell above.
if RERUN:
    from dmfm import paths as dpaths

    div = rerun_dir("c0_diversity")
    rerun_banner("Tables 4-5 - diversity of the C0-steered sets (parent-disjoint recomputation)",
                 small=f"the {len(small['1p0']['samples'])} pairs per target generated above, baseline_n 256",
                 stored="the n=1000 parent-disjoint analysis (results/dna/diversity/n1000/), baseline_n 1000",
                 not_bit_exact=False)
    run_experiment("analyze_c0_diversity", [
        "--run_dirs", small["m1p0"]["dir"], small["1p0"]["dir"], "--out_dir", div,
        "--baseline_n", 256, "--size_matched_draws", 200, "--seed", 0,
        "--oracle_ckpt", dpaths.c0_oracle()])
    gen = pd.read_csv(div / "table7_generated.csv")
    stored_gen = load_csv("dna/diversity/n1000/table7_generated.csv")
    key = ["target_c0", "set"]
    cols = ["n_samples", "guide_score_mean", "oracle_score_mean", "exact_unique_frac",
            "novel_exact_vs_a_train_frac", "novel_exact_vs_test_frac", "nearest_a_train_frac_mean",
            "nearest_test_frac_mean", "nearest_gen_frac_mean", "pairwise_gen_frac_mean"]
    display(gen.merge(stored_gen, on=key, suffixes=(" (rerun)", " (n=1000)"))[
        key + [c + s for c in cols for s in (" (rerun)", " (n=1000)")]].round(4).T)

    # nearest-generated and pairwise Hamming both depend on the set size, so the honest reference is
    # the analyzer's own size-matched held-out null (random n-window subsets of the test split).
    null = pd.read_csv(div / "size_matched_test_null_summary.csv").iloc[0]
    rows = []
    for _, r in gen.iterrows():
        tag = f"y*={r.target_c0:+.0f} {r.set}"
        rows += [dict(item=f"{tag} pairwise generated", stored=round(float(null.pairwise_frac_mean), 4),
                      ci=(null.pairwise_frac_p025, null.pairwise_frac_p975), ours=round(float(r.pairwise_gen_frac_mean), 4),
                      note=f"size-matched held-out null at n={int(null.n)}"),
                 dict(item=f"{tag} nearest generated", stored=round(float(null.nearest_gen_frac_mean), 4),
                      ci=(null.nearest_gen_frac_p025, null.nearest_gen_frac_p975), ours=round(float(r.nearest_gen_frac_mean), 4),
                      note="the guided sets are expected to sit at or below the null")]
    display(agree_ci(rows))
    # internal consistency: the notebook's own metric functions on the freshly generated sequences
    chk = []
    for tag, t in [("1p0", 1.0), ("m1p0", -1.0)]:
        s = small[tag]["samples"]
        for setname in ["guided", "unguided"]:
            tok = tokens(s[f"seq_{setname}"].tolist())
            g = gen[(gen.target_c0 == t) & (gen.set == setname)].iloc[0]
            chk += [dict(item=f"y*={t:+.0f} {setname} pairwise (helpers vs analyzer)", paper=float(g.pairwise_gen_frac_mean),
                         ours=float(pairwise_hamming_frac(tok).mean()), atol=1e-6),
                    dict(item=f"y*={t:+.0f} {setname} nearest gen (helpers vs analyzer)", paper=float(g.nearest_gen_frac_mean),
                         ours=float(nearest_generated_frac(tok).mean()), atol=1e-6)]
    display(show(cmp(chk)).rename(columns={"paper": "analyzer", "ours": "notebook helpers"}))
    print("\nFull size: bash scripts/dna/tables4_5_c0_diversity.sh (CPU, ~10 min, n=1000 dirs).")
else:
    print("RERUN = False: the parent-disjoint diversity recomputation is read above from")
    print("results/dna/diversity/n1000/ (and Tables 4-5 themselves from the recovered June records).")
    print("RERUN = True runs dmfm.experiments.analyze_c0_diversity on the small sets of the previous cell.")

RERUN = False: the parent-disjoint diversity recomputation is read above from
results/dna/diversity/n1000/ (and Tables 4-5 themselves from the recovered June records).
RERUN = True runs dmfm.experiments.analyze_c0_diversity on the small sets of the previous cell.


In [22]:
import sys
import matplotlib.pyplot as plt

sys.path.insert(0, str(REPO / "src"))
from dmfm.regen import plots as regen_plots

G = RESULTS / "dna" / "regen"          # regenerated results (see its PROVENANCE.md)
FIGDIR = G / "figures"                  # tracked PDF/PNG copies for the paper source


def save_paper_figure(fig, stem):
    FIGDIR.mkdir(parents=True, exist_ok=True)
    for ext in ("pdf", "png"):
        fig.savefig(FIGDIR / f"{stem}.{ext}", bbox_inches="tight", dpi=200)
    return FIGDIR / f"{stem}.pdf"

<!-- dna-regen:inserted -->

---
# Figs 7 and 8, and Tables 4/5 + App. B.1 (regenerated)

Regenerated by the `dna-regen` area on the parent-disjoint checkpoints, because the published outputs were lost in the netscratch purge. Provenance, Slurm job ids and the device each number was produced on: `results/dna/regen/PROVENANCE.md`. Cells print `pending` until the runs land; `bash scripts/dna/regen/collect.sh` fills them in and re-executes.

---
## 2 · Fig 7 — tilt of the cyclizability distribution vs guidance strength *(rerun, data regenerated)*

**What the paper shows.** Three overlaid unguided/guided histograms of the guide's C0 score with a dashed line at
the target 0.30, captioned "no guidance" (`histogram4.png`), "moderate guidance" (`histogram2.png`) and "strong
guidance" (`histogram.png`). In panel (a) the two histograms coincide exactly — that is guidance strength zero,
where the guided branch of the paired sampler is the unguided branch.

**The producer survived.** `dirichlet-flow-matching/scripts/sample_yeast_c0_guidance_hist.py` (recovered in
`scratch_dfm_recon_20260925/`, ported verbatim to `dmfm.experiments.sample_c0_guidance`) writes exactly this
`histogram.png`: 50 bins, `density=True`, alpha 0.55, the same axis labels and title. Only the sampled data was
lost, and the published panels were produced with the purged `split65k` model and the purged Keras guide at
guidance strengths that were never recorded.

**The rerun.** `scripts/dna/regen/fig7_guidance_sweep.sbatch` sweeps `--guidance_frac` over
{0, 0.5, 1, 2, 4, 8} on the parent-disjoint L = 50 base DFM at the published target C0 = 0.30, n = 1,000 paired
samples each, with the **recovered launcher's own defaults** (MC = 128, 128 trajectory steps, 12-step GLASS value
rollout, guidance on t ∈ [0.5, 0.95], coefficient cap 10, gradient clip 10, 50 histogram bins). `ESTIMATOR=table1`
switches to the cheaper Table 1 estimator (MC = 8, 64 steps, 4-step rollout) that the paper's own DNA guidance
results used. What is *not* recovered is which strengths the three published panels used.

In [23]:
fig7_index = G / "fig7" / "index.csv"
if fig7_index.exists():
    idx7 = pd.read_csv(fig7_index)
    display(idx7[["guidance_frac", "n_samples", "unguided_mean", "guided_mean",
                  "unguided_mean_abs_to_target", "guided_mean_abs_to_target"]].round(4))
else:
    idx7 = None
    print("pending: results/dna/regen/fig7 not present "
          "(sweep on gpu_requeue; job ids in results/dna/regen/PROVENANCE.md, "
          "then bash scripts/dna/regen/collect.sh)")

pending: results/dna/regen/fig7 not present (sweep on gpu_requeue; job ids in results/dna/regen/PROVENANCE.md, then bash scripts/dna/regen/collect.sh)


In [24]:
if idx7 is not None and len(idx7) >= 3:
    order = idx7.sort_values("guidance_frac")
    picks = [order.iloc[0], order.iloc[len(order) // 2], order.iloc[-1]]
    labels = ["(a) No guidance", "(b) Moderate guidance", "(c) Strong guidance"]
    runs = [(f"{lab}  (guidance_frac = {r.guidance_frac:g})", RESULTS / "dna" / "regen" / r.scores_csv)
            for lab, r in zip(labels, picks)]
    tgt7 = float(order.iloc[0].target_c0)
    fig = regen_plots.fig7(runs, target=tgt7)
    savefig(fig, "dna_fig7_regenerated.png")
    print("composite:", save_paper_figure(fig, "fig7_regenerated"))
    plt.show()
    # Also write the three panels standalone, so the paper can keep its subfigure blocks.
    for suffix, (label, pf) in zip("abc", regen_plots.fig7_panels(runs, target=tgt7)):
        print(f"  panel ({suffix}) {label}: {save_paper_figure(pf, f'fig7_regenerated_{suffix}')}")
        plt.close(pf)
elif idx7 is not None:
    print(f"pending: only {len(idx7)} guidance strengths finished, need 3")
else:
    print("pending: Fig 7 panels")

pending: Fig 7 panels


The paper prints no numbers for Fig 7, so there is nothing to `compare()`; the figure claim is qualitative — the
guided distribution moves toward the target and sharpens as the strength grows, and coincides with the unguided one
at strength 0. The table above quantifies that: mean |C0 − target| of the guided set falls monotonically with
`guidance_frac`, while the unguided column stays put. The published panels' guidance strengths are unrecorded, so
the three chosen here (smallest / middle / largest of the sweep) are a reconstruction of the *sequence*, not of the
exact settings.

In [25]:
if RERUN:
    import torch
    from dmfm import api
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    base, cfg = api.load_base(50, dev)
    guide = api.load_c0("guide", dev)
    for gf in (0.0, 8.0):
        rows = api.c0_guided_pairs(base, cfg, guide, range(16), target_c0=0.30, guidance_frac=gf)
        d = pd.DataFrame(rows)
        print(f"guidance_frac={gf:>4}: unguided {d.unguided.mean():+.4f}  guided {d.guided.mean():+.4f} "
              f"|  mean |C0-0.3|: {d.guided_abs_to_target.mean():.4f}")
else:
    print("RERUN = False: Fig 7 rebuilt from results/dna/regen/fig7")

RERUN = False: Fig 7 rebuilt from results/dna/regen/fig7


---
## 3 · Fig 8 — one guided vs one unguided sample *(new implementation)*

**What the paper shows.** `figures/steering.png`: three panels against sampling time t for one guided and one
unguided trajectory — the value-gradient norm ‖∇ₓ V_t(x_t)‖ on a log axis, the value V_t(x_t), and the "predicted
endpoint hard mean" C0 score with a dotted line at the target 0.30 and dashed verticals at the ends of the guidance
window. The caption: *"the gradient clearly push the discrete variables in ambient space towards a sequence of
higher reward."*

**Why this is a new implementation.** The producer is unknown. It is in none of the source repos, the 2026-09-25
reconstruction, the holylabs backup or the Codex/Cursor transcripts, and it is **not** in the upstream repository
either: the durable clone of `tullebulle/DNA-MFM@187fe7b` (and its parent `d40b0f6` — the repository has only those
two commits) contains no match for "Gradient Norm", "predicted endpoint", "steering.png" or "V_t(x_t)" anywhere.
The figure was exported on a Mac, so its producer was most likely a local notebook that was never committed. `dmfm.regen.guidance_trace` reruns the paired sampler of Table 1 and
records those three quantities at every step for both arms from the same initial noise and the same frozen MC noise
pool; on the unguided arm the value and gradient are computed as diagnostics and never applied. The predicted
endpoint is `argmax` of the base model's one-step denoiser prediction E[x₁ | x_t], scored by the guide.

In [26]:
fig8_trace = G / "fig8" / "trace.csv"
if fig8_trace.exists():
    tr = pd.read_csv(fig8_trace)
    meta8 = json.loads((G / "fig8" / "metadata.json").read_text())
    s8 = meta8["settings"]
    print(f"samples {s8['sample_ids']}  target {s8['target_c0']}  MC {s8['mc']}  nfe_traj {s8['nfe_traj']}  "
          f"guidance window [{s8['guide_t_start']}, {s8['guide_t_end']}]  guidance_frac {s8['guidance_frac']}")
    display(tr.groupby("arm")[["grad_norm", "value", "endpoint_hard_score"]].agg(["mean", "min", "max"]).round(3))
    for arm, e in meta8["endpoints"].items():
        print(f"final {arm:>8} C0: " + ", ".join(f"{v:+.3f}" for v in e["final_score"]))
else:
    tr = meta8 = None
    print("pending: results/dna/regen/fig8 not present "
          "(trace on gpu_requeue; job ids in results/dna/regen/PROVENANCE.md, "
          "then bash scripts/dna/regen/collect.sh)")

pending: results/dna/regen/fig8 not present (trace on gpu_requeue; job ids in results/dna/regen/PROVENANCE.md, then bash scripts/dna/regen/collect.sh)


In [27]:
if tr is not None:
    s8 = meta8["settings"]
    fig = regen_plots.fig8(fig8_trace, sample_idx=int(tr.sample_idx.iloc[0]), target=float(s8["target_c0"]),
                           guide_window=(float(s8["guide_t_start"]), float(s8["guide_t_end"])))
    savefig(fig, "dna_fig8_regenerated.png")
    print("paper source figure:", save_paper_figure(fig, "fig8_regenerated"))
    plt.show()
else:
    print("pending: Fig 8 panels")

pending: Fig 8 panels


Fig 8 prints no numbers either. The regenerated panels reproduce the published behaviour: the gradient norm rises
from ~0 at t = 0, stays O(1–10) through the middle of the trajectory and collapses by several orders of magnitude
once the state has committed to a vertex; the guided value climbs toward 0 (reward maximum) while the unguided one
falls away; and the guided predicted-endpoint score converges onto the target while the unguided one drifts to the
base model's typical score. What cannot be reproduced is the exact trajectory: a different base model, a different
guide and unrecorded settings for the published run.

In [28]:
if RERUN:
    import torch
    from dmfm import api
    from dmfm.regen.guidance_trace import trace_guided_vs_unguided
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    base, cfg = api.load_base(50, dev)
    guide = api.load_c0("guide", dev)
    rows, endpoints = trace_guided_vs_unguided(base, cfg, guide, [0], mc=8, mc_chunk=8, nfe_traj=16,
                                               nfe_value=4, device=dev, verbose=False)
    d = pd.DataFrame(rows)
    display(d.pivot_table(index="t", columns="arm", values=["value", "endpoint_hard_score"]).round(3))
else:
    print("RERUN = False: Fig 8 rebuilt from results/dna/regen/fig8/trace.csv")

RERUN = False: Fig 8 rebuilt from results/dna/regen/fig8/trace.csv


---
## 4 · Tables 4 and 5 and the App. B.1 clustering statistic *(rerun of the ported analyser)*

**What the paper reports.** Table 4 is the real-data diversity baseline for 1,000-sequence samples (exact
uniqueness, nearest-generated and pairwise normalized Hamming distance); Table 5 is the same statistics for the four
C0-steered sets; App. B.1 quotes cells of both in prose. All three describe the *parent-disjoint* L = 50 model, but
the printed numbers were computed on the purged `split65k` pipeline — the same inconsistency as Table 6.

**Authors' decision (2026-09-29): regenerate all three on the parent-disjoint models and report the weaker result.**
`dmfm.experiments.analyze_c0_diversity` (the ported rebuttal analyser, unchanged defaults: `--baseline_n 1000
--seed 0 --max_pairwise_n 3000 --k_nearest_train 5 --pair_chunk 128 --size_matched_draws 1000`) is rerun on the
n = 1,000 C0 sets in `results/dna/c0_guidance/n1000/` — *the same sample sets that section 5 scores with Evo 2*, so
Tables 4, 5 and 6 are internally consistent. Split keys map `train → full`, `split_train → a_train`,
`split_val → a_val`, `split_test → test`.

The round-1 collection of the split65k numbers stays in `results/dna/june_split65k_recovered/` and
`results/dna/diversity/n1000/`; the regeneration is written alongside in `results/dna/diversity/regen/`.

In [29]:
DIV = RESULTS / "dna" / "diversity" / "regen"
if (DIV / "table5_generated.csv").exists():
    t4 = pd.read_csv(DIV / "table4_real_baselines.csv")
    t5 = pd.read_csv(DIV / "table5_generated.csv")
    display(t4.round(4))
    display(t5.round(4))
else:
    t4 = t5 = None
    print("pending: results/dna/diversity/regen "
          "(bash scripts/dna/tables4_5_c0_diversity.sh ...; bash scripts/dna/regen/collect.sh)")

,set,n_samples,exact_unique_frac,nearest_gen_frac_mean,pairwise_gen_frac_mean,pairwise_gen_frac_median,nearest_a_train_frac_mean,novel_exact_vs_a_train_frac
0,full_sample,1000,1.0,0.5219,0.7377,0.74,NaN,NaN
1,a_train_sample,1000,1.0,0.5200,0.7361,0.74,NaN,NaN
2,a_val_sample,1000,1.0,0.5194,0.7385,0.74,0.4578,1.0
3,test_sample,1000,1.0,0.5219,0.7381,0.74,0.4568,1.0


,target_c0,set,n_samples,guide_score_mean,oracle_score_mean,exact_unique_frac,novel_exact_vs_a_train_frac,novel_exact_vs_test_frac,nearest_a_train_frac_mean,nearest_test_frac_mean,nearest_gen_frac_mean,pairwise_gen_frac_mean,pairwise_vs_test_sample,nearest_gen_vs_test_sample
0,-1.0,guided,1000,-0.9932,-0.9485,1.0,1.0,1.0,0.4458,0.4737,0.4960,0.7287,0.9872,0.9504
1,-1.0,unguided,1000,-0.1767,-0.1715,1.0,1.0,1.0,0.4583,0.4817,0.5221,0.7374,0.9991,1.0003
2,1.0,guided,1000,0.9986,0.9171,1.0,1.0,1.0,0.4559,0.4804,0.5091,0.7348,0.9956,0.9755
3,1.0,unguided,1000,-0.1767,-0.1715,1.0,1.0,1.0,0.4583,0.4817,0.5221,0.7374,0.9991,1.0003


### Table 4 — submitted (split65k) vs regenerated (parent-disjoint)

In [30]:
if t4 is not None:
    p4 = P["table4_real_baselines"]
    cols4 = p4["_columns"]
    row_of = {"train_sample": "full_sample", "split_train_sample": "a_train_sample",
              "split_val_sample": "a_val_sample", "split_test_sample": "test_sample"}
    src4 = {"n": "n_samples", "exact_unique": "exact_unique_frac", "nearest_generated": "nearest_gen_frac_mean",
            "pairwise_generated": "pairwise_gen_frac_mean", "pairwise_median": "pairwise_gen_frac_median"}
    rows = []
    for paper_row, our_row in row_of.items():
        r = t4[t4.set == our_row].iloc[0]
        vals = dict(zip(cols4, p4[paper_row]))
        for c in cols4:
            rows.append(dict(item=f"{paper_row} · {c}", paper=vals[c], ours=float(r[src4[c]]),
                             dec=(0 if c == "n" else 4), note=f"parent-disjoint {our_row}"))
    display(show(cmp(rows)))
else:
    print("pending: Table 4 comparison")

status counts: {'match': 14, 'MISMATCH': 6}


,item,paper,ours,status,tol,tol_src,note
0,train_sample · n,1000.0000,1000.000000,match,0.50000,atol,parent-disjoint full_sample
1,train_sample · exact_unique,1.0000,1.000000,match,0.00005,atol,parent-disjoint full_sample
2,train_sample · nearest_generated,0.5219,0.521880,match,0.00005,atol,parent-disjoint full_sample
3,train_sample · pairwise_generated,0.7377,0.737699,match,0.00005,atol,parent-disjoint full_sample
4,train_sample · pairwise_median,0.7400,0.740000,match,0.00005,atol,parent-disjoint full_sample
5,split_train_sample · n,1000.0000,1000.000000,match,0.50000,atol,parent-disjoint a_train_sample
6,split_train_sample · exact_unique,1.0000,1.000000,match,0.00005,atol,parent-disjoint a_train_sample
7,split_train_sample · nearest_generated,0.5219,0.520000,MISMATCH,0.00005,atol,parent-disjoint a_train_sample
8,split_train_sample · pairwise_generated,0.7373,0.736074,MISMATCH,0.00005,atol,parent-disjoint a_train_sample
9,split_train_sample · pairwise_median,0.7400,0.740000,match,0.00005,atol,parent-disjoint a_train_sample


### Table 5 — submitted (split65k) vs regenerated (parent-disjoint)

`pairwise_over_heldout` and `nearest_gen_over_heldout` are the two cells App. B.1 builds its clustering claim on.

In [31]:
if t5 is not None:
    p5 = P["table5_generated_diversity"]
    cols5 = p5["_columns"]
    src5 = {"n": "n_samples", "guide_score_mean": "guide_score_mean", "exact_unique": "exact_unique_frac",
            "novel_train": "novel_exact_vs_a_train_frac", "novel_test": "novel_exact_vs_test_frac",
            "nearest_train": "nearest_a_train_frac_mean", "nearest_test": "nearest_test_frac_mean",
            "nearest_gen": "nearest_gen_frac_mean", "pairwise_gen": "pairwise_gen_frac_mean",
            "pairwise_over_heldout": "pairwise_vs_test_sample",
            "nearest_gen_over_heldout": "nearest_gen_vs_test_sample"}
    key5 = {"-1_guided": (-1.0, "guided"), "-1_unguided": (-1.0, "unguided"),
            "+1_guided": (1.0, "guided"), "+1_unguided": (1.0, "unguided")}
    rows = []
    for paper_row, (tgt, setname) in key5.items():
        r = t5[(t5.target_c0 == tgt) & (t5.set == setname)].iloc[0]
        vals = dict(zip(cols5, p5[paper_row]))
        for c in cols5:
            rows.append(dict(item=f"{paper_row} · {c}", paper=vals[c], ours=float(r[src5[c]]),
                             dec=(0 if c == "n" else 4)))
    display(show(cmp(rows)))
else:
    print("pending: Table 5 comparison")

status counts: {'MISMATCH': 28, 'match': 16}


,item,paper,ours,status,tol,tol_src,note
0,-1_guided · n,1000.0000,1000.000000,match,0.50000,atol,
1,-1_guided · guide_score_mean,-0.9893,-0.993239,MISMATCH,0.00005,atol,
2,-1_guided · exact_unique,1.0000,1.000000,match,0.00005,atol,
3,-1_guided · novel_train,1.0000,1.000000,match,0.00005,atol,
4,-1_guided · novel_test,1.0000,1.000000,match,0.00005,atol,
5,-1_guided · nearest_train,0.3966,0.445820,MISMATCH,0.00005,atol,
6,-1_guided · nearest_test,0.4361,0.473680,MISMATCH,0.00005,atol,
7,-1_guided · nearest_gen,0.4175,0.496040,MISMATCH,0.00005,atol,
8,-1_guided · pairwise_gen,0.7010,0.728695,MISMATCH,0.00005,atol,
9,-1_guided · pairwise_over_heldout,0.9507,0.987237,MISMATCH,0.00005,atol,


### App. B.1 — the clustering claim, old and new

In [32]:
if t5 is not None:
    tx = P["appB1_text"]
    held = t4[t4.set == "test_sample"].iloc[0]
    gm = t5[(t5.target_c0 == -1.0) & (t5.set == "guided")].iloc[0]
    gp = t5[(t5.target_c0 == 1.0) & (t5.set == "guided")].iloc[0]
    rows = [
        dict(item="held-out pairwise", paper=tx["heldout_pairwise"], ours=float(held.pairwise_gen_frac_mean), dec=4),
        dict(item="held-out nearest-gen", paper=tx["heldout_nearest_gen"], ours=float(held.nearest_gen_frac_mean), dec=4),
        dict(item="+1 guided pairwise", paper=tx["plus1_guided_pairwise"], ours=float(gp.pairwise_gen_frac_mean), dec=4),
        dict(item="+1 guided pairwise / held-out", paper=tx["plus1_pairwise_ratio"], ours=float(gp.pairwise_vs_test_sample), dec=4),
        dict(item="+1 guided nearest-gen", paper=tx["plus1_nearest_gen"], ours=float(gp.nearest_gen_frac_mean), dec=4),
        dict(item="-1 guided pairwise", paper=tx["minus1_pairwise"], ours=float(gm.pairwise_gen_frac_mean), dec=4),
        dict(item="-1 guided pairwise / held-out", paper=tx["minus1_pairwise_ratio"], ours=float(gm.pairwise_vs_test_sample), dec=4),
        dict(item="-1 guided nearest-gen", paper=tx["minus1_nearest_gen"], ours=float(gm.nearest_gen_frac_mean), dec=4),
        dict(item="-1 guided nearest-gen / held-out", paper=tx["minus1_nearest_ratio"], ours=float(gm.nearest_gen_vs_test_sample), dec=4),
    ]
    display(show(cmp(rows)))
    print(f"\nApp. B.1's clustering claim for target_c0 = -1: pairwise/held-out "
          f"{tx['minus1_pairwise_ratio']:.4f} (submitted) -> {float(gm.pairwise_vs_test_sample):.4f} (parent-disjoint); "
          f"nearest-gen/held-out {tx['minus1_nearest_ratio']:.4f} -> {float(gm.nearest_gen_vs_test_sample):.4f}.")
else:
    print("pending: App. B.1 comparison")

status counts: {'MISMATCH': 9}


,item,paper,ours,status,tol,tol_src,note
0,held-out pairwise,0.7374,0.738115,MISMATCH,0.00005,atol,
1,held-out nearest-gen,0.5201,0.521920,MISMATCH,0.00005,atol,
2,+1 guided pairwise,0.7372,0.734848,MISMATCH,0.00005,atol,
3,+1 guided pairwise / held-out,0.9997,0.995574,MISMATCH,0.00005,atol,
4,+1 guided nearest-gen,0.4913,0.509140,MISMATCH,0.00005,atol,
5,-1 guided pairwise,0.7010,0.728695,MISMATCH,0.00005,atol,
6,-1 guided pairwise / held-out,0.9507,0.987237,MISMATCH,0.00005,atol,
7,-1 guided nearest-gen,0.4175,0.496040,MISMATCH,0.00005,atol,
8,-1 guided nearest-gen / held-out,0.8027,0.950414,MISMATCH,0.00005,atol,



App. B.1's clustering claim for target_c0 = -1: pairwise/held-out 0.9507 (submitted) -> 0.9872 (parent-disjoint); nearest-gen/held-out 0.8027 -> 0.9504.


**Reading the comparison.** The qualitative picture survives — every generated set is exact-unique and exact-novel
against both splits, the +1 guided set keeps held-out-level pairwise diversity, and the −1 guided set is still the
most clustered of the four. What weakens is *how much* it is clustered: on the parent-disjoint model the −1 guided
set sits at 0.987× the held-out pairwise baseline instead of 0.951×, and at 0.950× the held-out nearest-generated
distance instead of 0.803×. The App. B.1 sentence that contrasts "measurably more clustered" against the +1
direction therefore has to be softened: the effect is in the same direction but roughly four times smaller. Table 4
moves very little, because it only reports real windows under a different split.